# GOV-01 E8 specialist threshold calibration — Kaggle version

This notebook does not train again. It reuses E8 and selects the best validation confidence cutoff for pothole and manhole cover. Attach two inputs before running: the expansion dataset ZIP and the 56 MB E8 output ZIP. The protected test split is never loaded.


In [ ]:
# Locate the attached expansion dataset and E8 result ZIP.
from pathlib import Path
import csv
import json
import shutil
import zipfile

DATASET_FOLDER = 'v2_multilabel_n_rdd_expansion'
E8_OUTPUT_NAME = 'v2_multilabel_e8_pothole_manhole_specialist_output.zip'
MODEL_NAME = 'e8_pothole_manhole_specialist_best.keras'
SPECIALIST_CONDITIONS = ('pothole', 'manhole_cover')
IMAGE_SIZE = (320, 320)
BATCH_SIZE = 16
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
LOCAL_DATA_ROOT = WORK_ROOT / DATASET_FOLDER
LOCAL_MODEL_ROOT = WORK_ROOT / 'v2_multilabel_e8_calibration_model'
MODEL_PATH = LOCAL_MODEL_ROOT / MODEL_NAME
OUTPUT_DIRECTORY = WORK_ROOT / 'v2_multilabel_e8_threshold_calibration_output'

def has_training_layout(root):
    return all((root / split / 'images').is_dir() and (root / split / 'labels.csv').is_file() for split in ('train', 'validation'))

attached_folders = [path for path in INPUT_ROOT.rglob(DATASET_FOLDER) if path.is_dir() and has_training_layout(path)]
attached_archives = list(INPUT_ROOT.rglob(f'{DATASET_FOLDER}.zip'))
if len(attached_folders) == 1:
    DATA_ROOT, DATA_ARCHIVE = attached_folders[0], None
elif len(attached_archives) == 1:
    DATA_ROOT, DATA_ARCHIVE = LOCAL_DATA_ROOT, attached_archives[0]
else:
    raise FileNotFoundError('Attach exactly one Kaggle Dataset containing v2_multilabel_n_rdd_expansion.zip or the extracted folder.')

e8_archives = list(INPUT_ROOT.rglob(E8_OUTPUT_NAME))
e8_models = list(INPUT_ROOT.rglob(MODEL_NAME))
if len(e8_archives) == 1:
    E8_ARCHIVE, ATTACHED_MODEL_PATH = e8_archives[0], None
elif len(e8_models) == 1:
    # Kaggle normally unzips uploaded Dataset archives, so accept the extracted best model too.
    E8_ARCHIVE, ATTACHED_MODEL_PATH = None, e8_models[0]
else:
    raise FileNotFoundError('Attach exactly one E8 output ZIP or one extracted e8_pothole_manhole_specialist_best.keras file through a private Kaggle Dataset.')
print('Dataset:', DATA_ARCHIVE or DATA_ROOT)
print('E8 output:', E8_ARCHIVE or ATTACHED_MODEL_PATH)
print('Protected test is deliberately not loaded.')


In [ ]:
# Reuse validation data if available; otherwise extract validation only. Extract the E8 best model.
def validation_is_complete(dataset_root):
    split_root, labels_path = dataset_root / 'validation', dataset_root / 'validation' / 'labels.csv'
    if not labels_path.is_file() or not (split_root / 'images').is_dir():
        return False
    with labels_path.open(newline='', encoding='utf-8') as source:
        rows = list(csv.DictReader(source))
    return bool(rows) and all((split_root / Path(row['materialized_image'].replace(chr(92), '/'))).is_file() for row in rows)

def assert_safe_members(members, destination_root):
    destination_root = destination_root.resolve()
    for member in members:
        destination = (destination_root / member.filename).resolve()
        if destination != destination_root and destination_root not in destination.parents:
            raise RuntimeError(f'Unsafe ZIP member: {member.filename}')

if DATA_ARCHIVE is not None and not validation_is_complete(LOCAL_DATA_ROOT):
    if LOCAL_DATA_ROOT.exists(): shutil.rmtree(LOCAL_DATA_ROOT)
    prefix = f'{DATASET_FOLDER}/validation/'
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:
        members = [member for member in archive.infolist() if member.filename.startswith(prefix)]
        if not members: raise RuntimeError('Dataset ZIP has no validation folder at the expected location.')
        assert_safe_members(members, WORK_ROOT)
        archive.extractall(WORK_ROOT, members=members)
if not validation_is_complete(DATA_ROOT):
    raise RuntimeError(f'Validation data is incomplete under {DATA_ROOT}')

if ATTACHED_MODEL_PATH is not None:
    MODEL_PATH = ATTACHED_MODEL_PATH
elif not MODEL_PATH.is_file():
    LOCAL_MODEL_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(E8_ARCHIVE) as archive:
        member = archive.getinfo(MODEL_NAME)
        assert_safe_members([member], LOCAL_MODEL_ROOT)
        archive.extract(member, LOCAL_MODEL_ROOT)
if not MODEL_PATH.is_file(): raise RuntimeError('Could not extract the E8 best model.')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
print('E8 model ready:', MODEL_PATH)


In [ ]:
# Read validation labels and generate E8 probabilities. GPU is helpful but not required.
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score, precision_recall_fscore_support

print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
split_root = DATA_ROOT / 'validation'
paths, labels, masks = [], [], []
with (split_root / 'labels.csv').open(newline='', encoding='utf-8') as source:
    for row in csv.DictReader(source):
        image_path = split_root / Path(row['materialized_image'].replace(chr(92), '/'))
        if not image_path.is_file(): raise FileNotFoundError(f'Missing validation image: {image_path}')
        label_row, mask_row = [], []
        for condition in SPECIALIST_CONDITIONS:
            known, value = int(row[f'{condition}_known']), row[f'{condition}_present']
            if known not in (0, 1) or (known == 0 and value != '') or (known == 1 and value not in ('0', '1')):
                raise ValueError(f'Invalid masked label: {row["record_id"]} / {condition}')
            label_row.append(float(value) if known else 0.0); mask_row.append(float(known))
        if any(mask_row):
            paths.append(str(image_path)); labels.append(label_row); masks.append(mask_row)
validation_paths = np.asarray(paths)
validation_labels = np.asarray(labels, dtype='float32')
validation_masks = np.asarray(masks, dtype='float32')

def load_image(path):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    return tf.cast(tf.image.resize(image, IMAGE_SIZE), tf.float32)

validation_images = tf.data.Dataset.from_tensor_slices(validation_paths).map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
best_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
validation_probabilities = best_model.predict(validation_images, verbose=1)
print('Prediction shape:', validation_probabilities.shape)


In [ ]:
# Select each specialist cutoff from 0.05 to 0.95 using validation F1 only.
import matplotlib.pyplot as plt

THRESHOLD_GRID = np.round(np.arange(0.05, 0.96, 0.01), 2)
DEFAULT_THRESHOLD = 0.50
calibration = {}
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for index, condition in enumerate(SPECIALIST_CONDITIONS):
    known = validation_masks[:, index].astype(bool)
    actual, probabilities = validation_labels[known, index].astype(int), validation_probabilities[known, index]
    scores = np.asarray([f1_score(actual, probabilities >= threshold, zero_division=0) for threshold in THRESHOLD_GRID])
    best_score = scores.max()
    tied = np.flatnonzero(np.isclose(scores, best_score))
    best_index = min(tied, key=lambda item: (abs(float(THRESHOLD_GRID[item]) - DEFAULT_THRESHOLD), float(THRESHOLD_GRID[item])))
    selected = float(THRESHOLD_GRID[best_index])
    default_precision, default_recall, default_f1, _ = precision_recall_fscore_support(actual, probabilities >= DEFAULT_THRESHOLD, average='binary', zero_division=0)
    selected_precision, selected_recall, selected_f1, _ = precision_recall_fscore_support(actual, probabilities >= selected, average='binary', zero_division=0)
    calibration[condition] = {'known_images': int(len(actual)), 'positive_images': int(actual.sum()), 'default_threshold': DEFAULT_THRESHOLD, 'default_precision': float(default_precision), 'default_recall': float(default_recall), 'default_f1': float(default_f1), 'selected_threshold': selected, 'selected_precision': float(selected_precision), 'selected_recall': float(selected_recall), 'selected_f1': float(selected_f1), 'f1_change': float(selected_f1-default_f1)}
    axes[index].plot(THRESHOLD_GRID, scores); axes[index].axvline(DEFAULT_THRESHOLD, color='gray', linestyle='--', label='default'); axes[index].axvline(selected, color='green', linestyle='--', label=f'selected {selected:.2f}')
    axes[index].set_title(condition); axes[index].set_xlabel('confidence cutoff'); axes[index].set_ylabel('validation F1'); axes[index].legend()
figure.tight_layout(); figure.savefig(OUTPUT_DIRECTORY / 'e8_threshold_calibration_curves.png', dpi=150)
default_macro = float(np.mean([item['default_f1'] for item in calibration.values()]))
calibrated_macro = float(np.mean([item['selected_f1'] for item in calibration.values()]))
result = {'experiment': 'v2_multilabel_e8_threshold_calibration', 'selection_data': 'validation only', 'protected_test_loaded': False, 'default_specialist_macro_f1': default_macro, 'calibrated_specialist_macro_f1': calibrated_macro, 'macro_f1_change': calibrated_macro-default_macro, 'per_condition': calibration}
(OUTPUT_DIRECTORY / 'e8_threshold_calibration.json').write_text(json.dumps(result, indent=2) + '\n')
with (OUTPUT_DIRECTORY / 'e8_threshold_calibration.csv').open('w', newline='', encoding='utf-8') as destination:
    writer = csv.DictWriter(destination, fieldnames=['condition', *next(iter(calibration.values())).keys()]); writer.writeheader()
    for condition, values in calibration.items(): writer.writerow({'condition': condition, **values})
for condition, values in calibration.items(): print(f'{condition:16} cutoff={values["selected_threshold"]:.2f}  F1: {values["default_f1"]:.3f} -> {values["selected_f1"]:.3f}  change={values["f1_change"]:+.3f}')
print(f'\nSpecialist macro F1: {default_macro:.4f} -> {calibrated_macro:.4f}  change={calibrated_macro-default_macro:+.4f}')
print('Protected test remains untouched.')


In [ ]:
# Package the small calibration result. Download it from Kaggle Output after the run.
result_zip = WORK_ROOT / 'v2_multilabel_e8_threshold_calibration_output.zip'
if result_zip.exists(): result_zip.unlink()
shutil.make_archive(str(result_zip.with_suffix('')), 'zip', OUTPUT_DIRECTORY)
print('Completed output ZIP:', result_zip)
